# Getting Started with Strands Agents for Financial Services

This lab introduces Strands Agents through the lens of financial services. You will build an AI agent with tools commonly needed by FSI teams — loan calculations, stock lookups, and FX rate checks.

## Overview

In this lab, you will:
- Understand the core concepts of Strands Agents
- Create your first AI agent with built-in tools
- Build custom tools for specific use cases
- Explore conversation history and agent memory
- Learn best practices for agent development


## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

⚠️ **Important**: Enable Nova Pro model access in the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home#/modelaccess) if you haven't already.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich

In [1]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## What are Strands Agents?

Strands Agents is a Python framework that simplifies the creation of AI agents with tool integration capabilities. Key features include:

- **Simple Agent Creation**: Easy-to-use API for creating AI agents with minimal code
- **Built-in Tools**: Pre-built tools like calculators, web search, and more
- **Custom Tool Support**: Create your own tools with simple Python functions
- **Conversation Memory**: Automatic conversation history management
- **Model Flexibility**: Support for various language models including models in Amazon Bedrock and OpenAI

Strands Agents provides a foundation for building sophisticated AI applications that can interact with external systems and perform complex tasks.


## Building Financial Tools

Let's create three custom tools that a financial services agent would need:

1. **Loan Calculator** — Calculate mortgage/loan repayments
2. **Stock Lookup** — Get current stock prices (ASX/US)
3. **FX Rate** — Currency conversion rates

In [2]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
import yfinance as yf


@tool
def loan_calculator(principal: float, annual_rate: float, years: int) -> str:
    """Calculate monthly loan/mortgage repayment using standard amortization formula.

    Args:
        principal: Loan amount in dollars
        annual_rate: Annual interest rate as percentage (e.g., 6.2 for 6.2%)
        years: Loan term in years
    """
    monthly_rate = (annual_rate / 100) / 12
    num_payments = years * 12
    if monthly_rate == 0:
        monthly_payment = principal / num_payments
    else:
        monthly_payment = principal * (monthly_rate * (1 + monthly_rate)**num_payments) / ((1 + monthly_rate)**num_payments - 1)
    total_paid = monthly_payment * num_payments
    total_interest = total_paid - principal
    return (
        f"Loan: ${principal:,.2f} at {annual_rate}% over {years} years"
        f"Monthly repayment: ${monthly_payment:,.2f}"
        f"Total interest: ${total_interest:,.2f}"
        f"Total paid: ${total_paid:,.2f}"
    )


@tool
def stock_lookup(ticker: str) -> str:
    """Look up current stock price for ASX or US equities using live market data.

    Args:
        ticker: Stock ticker symbol (e.g., CBA.AX, BHP.AX, AAPL)
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        price = info.get("currentPrice") or info.get("regularMarketPrice", "N/A")
        prev_close = info.get("previousClose", 0)
        name = info.get("shortName", ticker)
        currency = info.get("currency", "")
        if price != "N/A" and prev_close:
            change_pct = ((price - prev_close) / prev_close) * 100
            direction = "▲" if change_pct > 0 else "▼"
            return f"{name} ({ticker}): ${price:.2f} {currency} {direction} {abs(change_pct):.1f}%"
        return f"{name} ({ticker}): ${price} {currency}"
    except Exception as e:
        return f"Error looking up {ticker}: {e}"


@tool
def fx_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    """Get foreign exchange rate and convert currency amount using live rates.

    Args:
        from_currency: Source currency code (e.g., AUD, USD, GBP)
        to_currency: Target currency code (e.g., USD, AUD, EUR)
        amount: Amount to convert (default 1.0)
    """
    try:
        pair = f"{from_currency}{to_currency}=X"
        data = yf.Ticker(pair)
        rate = data.info.get("regularMarketPrice", None)
        if rate:
            converted = amount * rate
            return f"{amount:,.2f} {from_currency} = {converted:,.2f} {to_currency} (rate: {rate:.4f})"
        return f"Could not fetch rate for {from_currency}/{to_currency}"
    except Exception as e:
        return f"Error: {e}"


print("✅ Financial tools defined: loan_calculator, stock_lookup (live), fx_rate (live)")


✅ Financial tools defined: loan_calculator, stock_lookup (live), fx_rate (live)


## Creating Your FSI Agent

Now let's create an agent with these financial tools. The agent will decide which tool to use based on your question.

In [3]:
# Create the FSI agent
fsi_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a financial services assistant supporting banking and fintech operations 
    in Australia. You provide concise, accurate 
    financial calculations and market data. Always show your working and cite the tools used.""",
    tools=[loan_calculator, stock_lookup, fx_rate, calculator],
)

fsi_agent("What's the monthly repayment on a $750,000 mortgage at 6.2% over 30 years?")

<thinking> To determine the monthly repayment on a $750,000 mortgage at 6.2% over 30 years, I need to use the loan_calculator tool. The principal amount is $750,000, the annual interest rate is 6.2%, and the loan term is 30 years. </thinking>

Tool #1: loan_calculator
The monthly repayment on a $750,000 mortgage at 6.2% over 30 years is $4,593.52. The total interest paid over the life of the loan will be $903,666.24, and the total amount paid will be $1,653,666.24. 

This calculation was performed using the loan_calculator tool.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The monthly repayment on a $750,000 mortgage at 6.2% over 30 years is $4,593.52. The total interest paid over the life of the loan will be $903,666.24, and the total amount paid will be $1,653,666.24. \n\nThis calculation was performed using the loan_calculator tool.'}], 'metadata': {'usage': {'inputTokens': 1995, 'outputTokens': 95, 'totalTokens': 2090}, 'metrics': {'latencyMs': 1107, 'timeToFirstByteMs': 406}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'loan_calculator': ToolMetrics(tool={'toolUseId': 'tooluse_FUMfG9oGgGuKcDICHeNBwE', 'name': 'loan_calculator', 'input': {'principal': 750000, 'years': 30, 'annual_rate': 6.2}}, call_count=1, success_count=1, error_count=0, total_time=0.0022399425506591797)}, cycle_durations=[2.098144054412842, 1.157909870147705], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='cab9ce09-f9b4-47d0-afb9-f54b6eeb7415', u

In [4]:
fsi_agent("What's the current price of CBA and how does it compare to Westpac?")

<thinking> To determine the current price of CBA (Commonwealth Bank of Australia) and Westpac, I need to use the stock_lookup tool. I will first get the current price of CBA and then get the current price of Westpac. After obtaining both prices, I will compare them. </thinking> 
Tool #2: stock_lookup

Tool #3: stock_lookup
The current price of CBA (Commonwealth Bank of Australia) is $160.85 AUD, and the current price of Westpac is $34.65 AUD. 

Comparing the two, CBA is significantly higher in price than Westpac. 

These prices were obtained using the stock_lookup tool.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The current price of CBA (Commonwealth Bank of Australia) is $160.85 AUD, and the current price of Westpac is $34.65 AUD. \n\nComparing the two, CBA is significantly higher in price than Westpac. \n\nThese prices were obtained using the stock_lookup tool.'}], 'metadata': {'usage': {'inputTokens': 2298, 'outputTokens': 66, 'totalTokens': 2364}, 'metrics': {'latencyMs': 785, 'timeToFirstByteMs': 392}}}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'loan_calculator': ToolMetrics(tool={'toolUseId': 'tooluse_FUMfG9oGgGuKcDICHeNBwE', 'name': 'loan_calculator', 'input': {'principal': 750000, 'years': 30, 'annual_rate': 6.2}}, call_count=1, success_count=1, error_count=0, total_time=0.0022399425506591797), 'stock_lookup': ToolMetrics(tool={'toolUseId': 'tooluse_5he2xUxjjhRYkJW9ChghzE', 'name': 'stock_lookup', 'input': {'ticker': 'CBA.AX'}}, call_count=2, success_count=2, error_count=0, total_time=2

In [5]:
fsi_agent("Convert $1,000,000 AUD to USD and EUR")

<thinking> To convert $1,000,000 AUD to USD and EUR, I need to use the fx_rate tool. I will first convert AUD to USD and then convert AUD to EUR. </thinking> 
Tool #4: fx_rate

Tool #5: fx_rate
$1,000,000 AUD converts to approximately $712,758.36 USD and €613,300.00 EUR.

These conversions were performed using the fx_rate tool.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '$1,000,000 AUD converts to approximately $712,758.36 USD and €613,300.00 EUR.\n\nThese conversions were performed using the fx_rate tool.'}], 'metadata': {'usage': {'inputTokens': 2610, 'outputTokens': 53, 'totalTokens': 2663}, 'metrics': {'latencyMs': 899, 'timeToFirstByteMs': 516}}}, metrics=EventLoopMetrics(cycle_count=6, tool_metrics={'loan_calculator': ToolMetrics(tool={'toolUseId': 'tooluse_FUMfG9oGgGuKcDICHeNBwE', 'name': 'loan_calculator', 'input': {'principal': 750000, 'years': 30, 'annual_rate': 6.2}}, call_count=1, success_count=1, error_count=0, total_time=0.0022399425506591797), 'stock_lookup': ToolMetrics(tool={'toolUseId': 'tooluse_5he2xUxjjhRYkJW9ChghzE', 'name': 'stock_lookup', 'input': {'ticker': 'CBA.AX'}}, call_count=2, success_count=2, error_count=0, total_time=2.59273099899292), 'fx_rate': ToolMetrics(tool={'toolUseId': 'tooluse_WZ4waVDlWzGRrcffhnqAKq', 'name': 'fx_rate', 'inpu

## Understanding the Agent Loop

Let's examine how the agent processes requests — which tools it called, what inputs it provided, and what results it received. This visibility is critical for FSI compliance (audit trail).

In [7]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {fsi_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in fsi_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], text[-1] if text else "",
        tool_name[-1] if tool_name else "",
        json.dumps(tool_input[-1], indent=2) if tool_input else "",
        (json.dumps(tool_result[-1], indent=2)[:500]) if tool_result else ""
    )

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 6

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                      ┃ Tool Name       ┃ Tool Input                ┃ Tool Result               ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ What's the monthly        │                 │                           │                           │
│           │ repayment on a $750,000   │                 │                           │                           │
│           │ mortgage at 6.2% over 30  │                 │                           │                           │
│           │ years?                    │                 │                           │                           │
├───────────┼───────────────────────────┼─────────────────┼───────────────────────────┼───────────────────────────┤
│ assistant │ <thinking> To determine   │ loan_calculator │ {                         │                           │
│           │ the monthly repayment on  │                 │   "principal": 750000,    │                           │
│           │ a $750,000 mortgage at    │                 │   "years": 30,            │                           │
│           │ 6.2% over 30 years, I     │                 │   "annual_rate": 6.2      │                           │
│           │ need to use the           │                 │ }                         │                           │
│           │ loan_calculator tool. The │                 │                           │                           │
│           │ principal amount is       │                 │                           │                           │
│           │ $750,000, the annual      │                 │                           │                           │
│           │ interest rate is 6.2%,    │                 │                           │                           │
│           │ and the loan term is 30   │                 │                           │                           │
│           │ years. </thinking>        │                 │                           │                           │
│           │                           │                 │                           │                           │
├───────────┼───────────────────────────┼─────────────────┼───────────────────────────┼───────────────────────────┤
│ user      │                           │                 │                           │ {                         │
│           │                           │                 │                           │   "text": "Loan:          │
│           │                           │                 │                           │ $750,000.00 at 6.2% over  │
│           │                           │                 │                           │ 30 yearsMonthly           │
│           │                           │                 │                           │ repayment: $4,593.52Total │
│           │                           │                 │                           │ interest:                 │
│           │                           │                 │                           │ $903,666.24Total paid:    │
│           │                           │                 │                           │ $1,653,666.24"            │
│           │                           │                 │                           │ }                         │
├───────────┼───────────────────────────┼─────────────────┼───────────────────────────┼───────────────────────────┤
│ assistant │ The monthly repayment on  │                 │                           │                           │
│           │ a $750,000 mortgage at    │                 │                           │                           │
│           │ 6.2% over 30 years is     │               

## Conversation Memory (Within Session)

The agent remembers context within a session. This is useful for follow-up questions — like a real conversation with a financial advisor.

In [8]:
fsi_agent("If the rate drops to 5.5%, how much would I save per month on that mortgage?")

<thinking> To determine the monthly savings if the interest rate drops to 5.5%, I need to recalculate the monthly repayment at the new rate and then compare it to the original monthly repayment. The original repayment was $4,593.52 at 6.2%. I will use the loan_calculator tool to find the new monthly repayment at 5.5%. </thinking> 
Tool #6: loan_calculator
The monthly repayment on a $750,000 mortgage at 5.5% over 30 years is $4,258.42. 

Comparing this to the original monthly repayment of $4,593.52 at 6.2%, you would save approximately $335.10 per month. 

This calculation was performed using the loan_calculator tool.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The monthly repayment on a $750,000 mortgage at 5.5% over 30 years is $4,258.42. \n\nComparing this to the original monthly repayment of $4,593.52 at 6.2%, you would save approximately $335.10 per month. \n\nThis calculation was performed using the loan_calculator tool.'}], 'metadata': {'usage': {'inputTokens': 2902, 'outputTokens': 90, 'totalTokens': 2992}, 'metrics': {'latencyMs': 1147, 'timeToFirstByteMs': 489}}}, metrics=EventLoopMetrics(cycle_count=8, tool_metrics={'loan_calculator': ToolMetrics(tool={'toolUseId': 'tooluse_kTK0u0i9lTuxaNS6K4thDz', 'name': 'loan_calculator', 'input': {'principal': 750000, 'years': 30, 'annual_rate': 5.5}}, call_count=2, success_count=2, error_count=0, total_time=0.005753040313720703), 'stock_lookup': ToolMetrics(tool={'toolUseId': 'tooluse_5he2xUxjjhRYkJW9ChghzE', 'name': 'stock_lookup', 'input': {'ticker': 'CBA.AX'}}, call_count=2, success_count=2, error_count=

In [9]:
fsi_agent("What have we discussed so far?")

Here's a summary of what we've discussed so far:

1. **Monthly Mortgage Repayment:**
   - Original calculation: $750,000 mortgage at 6.2% over 30 years results in a monthly repayment of $4,593.52.
   - New calculation with a reduced rate of 5.5%: Monthly repayment is $4,258.42, resulting in a monthly savings of approximately $335.10.

2. **Stock Prices:**
   - Current price of CBA (Commonwealth Bank of Australia) is $160.85 AUD.
   - Current price of Westpac is $34.65 AUD.

3. **Currency Conversion:**
   - $1,000,000 AUD converts to approximately $712,758.36 USD.
   - $1,000,000 AUD converts to approximately €613,300.00 EUR.

All calculations and data were obtained using the provided tools: loan_calculator, stock_lookup, and fx_rate.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's a summary of what we've discussed so far:\n\n1. **Monthly Mortgage Repayment:**\n   - Original calculation: $750,000 mortgage at 6.2% over 30 years results in a monthly repayment of $4,593.52.\n   - New calculation with a reduced rate of 5.5%: Monthly repayment is $4,258.42, resulting in a monthly savings of approximately $335.10.\n\n2. **Stock Prices:**\n   - Current price of CBA (Commonwealth Bank of Australia) is $160.85 AUD.\n   - Current price of Westpac is $34.65 AUD.\n\n3. **Currency Conversion:**\n   - $1,000,000 AUD converts to approximately $712,758.36 USD.\n   - $1,000,000 AUD converts to approximately €613,300.00 EUR.\n\nAll calculations and data were obtained using the provided tools: loan_calculator, stock_lookup, and fx_rate."}], 'metadata': {'usage': {'inputTokens': 3003, 'outputTokens': 244, 'totalTokens': 3247}, 'metrics': {'latencyMs': 2230, 'timeToFirstByteMs': 480}}}, met

## Memory Limitation: New Session = Blank Slate

If we create a new agent instance, it forgets everything. This is a key limitation we'll solve in Lab 06 with AgentCore Memory.

In [10]:
# New agent = new session = no memory
fresh_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a financial services assistant. Provide concise responses.",
    tools=[loan_calculator, stock_lookup, fx_rate, calculator],
)

fresh_agent("What mortgage rate were we discussing?")

<thinking> I need to recall the specific mortgage rate that was previously discussed. However, I don't have access to past conversation history or context. Therefore, I cannot directly recall the mortgage rate. I should inform the user that I don't have this information. </thinking>

I'm sorry, but I don't have access to past conversation history. Could you please provide me with the mortgage rate we were discussing?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "<thinking> I need to recall the specific mortgage rate that was previously discussed. However, I don't have access to past conversation history or context. Therefore, I cannot directly recall the mortgage rate. I should inform the user that I don't have this information. </thinking>\n\nI'm sorry, but I don't have access to past conversation history. Could you please provide me with the mortgage rate we were discussing?"}], 'metadata': {'usage': {'inputTokens': 1736, 'outputTokens': 89, 'totalTokens': 1825}, 'metrics': {'latencyMs': 1017, 'timeToFirstByteMs': 586}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[1.1847422122955322], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='db88f5b0-b47c-48fe-ac6a-47d389b9a7c2', usage={'inputTokens': 1736, 'outputTokens': 89, 'totalTokens': 1825})], usage={'inputTokens': 1736, 'outputTokens': 89, 

## Summary

In this lab, you:

- ✅ Built custom FSI tools (loan calculator, stock lookup, FX rates)
- ✅ Created an agent that autonomously selects the right tool
- ✅ Examined the agent loop (audit trail for compliance)
- ✅ Explored session-based memory and its limitations

### Next Steps

In the following labs, we'll enhance this agent with:
- **Lab 01**: Code Interpreter for dynamic fraud analysis and risk calculations
- **Lab 02**: Browser automation for regulatory monitoring
- **Lab 04**: Deploy tools as managed services (AgentCore Runtime)
- **Lab 05**: Full observability and audit trails
- **Lab 06**: Persistent memory across sessions (remember client preferences)